# Employee Attrition Prediction

## Internship Project - Week 2

### Author
Naman Arora

### Objective
Build a Machine Learning model to predict whether an employee is likely to leave the company based on HR analytics data.

# Task 1 - Data Loading & Exploration

In [2]:
import numpy as numpy
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler


In [3]:
df = pd.read_csv("../WA_Fn-UseC_-HR-Employee-Attrition.csv")

print(df.head(10))
print(df.shape)

# checking how many employees have left the company
print(df["Attrition"].value_counts())

# calculating the percentage of employees who have left the company
attrition_percentage = (df["Attrition"].value_counts()["Yes"] / len(df)) * 100
print(f"Percentage of employees who have left the company: {attrition_percentage:.2f}%")


# printing columns that are numeric 
numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns
print(numeric_columns)

# printing columns that are actegorical 
categorical_columns = df.select_dtypes(include=["object"]).columns
print(categorical_columns)

print(f"Total number of numerical columns: {len(numeric_columns)}")
print(f"Total number of categorical columns: {len(categorical_columns)}")

# observation that we made about the attrition rate 
print("Observation: The attrition rate is relatively low, with only a small percentage of employees leaving the company.\nThis suggests that the company may have a stable work environment and good engagement practices that help retain employees.\nHowever, it is important to further analyze the data to identify any potential factors that may contribute to employee attrition and address them proactively.")

   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   
5   32        No  Travel_Frequently       1005  Research & Development   
6   59        No      Travel_Rarely       1324  Research & Development   
7   30        No      Travel_Rarely       1358  Research & Development   
8   38        No  Travel_Frequently        216  Research & Development   
9   36        No      Travel_Rarely       1299  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8      

/var/folders/1n/1fwqfjps4v5cmtm5yhyw4kmm0000gn/T/ipykernel_2296/604444428.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=["object"]).columns


### Observation

The dataset is imbalanced because the number of employees who stayed with the company is significantly higher than the number of employees who left. This imbalance should be considered during model training to avoid bias toward the majority class.

# Task 2 - Data Cleaning & Preprocessing

In [4]:
print(df.isnull().sum())

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

### Observation

The dataset contains no missing (null) values in any column. Since all columns have complete data, no missing value treatment such as deletion or imputation is required.

In [5]:
'''
we check constant and identifier columns, standard way   
'''
constant_columns = []

for col in df.columns:
    if df[col].nunique() == 1:
        constant_columns.append(col)

print("Constant Columns:")
print(constant_columns)


identifier_columns = []

for col in df.columns:
    if df[col].nunique() == len(df):
        identifier_columns.append(col)

print("Possible Identifier Columns:")
print(identifier_columns)


# dropping constant and identifier columns
columns_to_drop = constant_columns + identifier_columns

print("Columns to Drop:")
print(columns_to_drop)

df = df.drop(columns=columns_to_drop)

print("Shape before dropping:", (1470, 35))   # or use the shape you noted earlier
print("Shape after dropping:", df.shape)


Constant Columns:
['EmployeeCount', 'Over18', 'StandardHours']
Possible Identifier Columns:
['EmployeeNumber']
Columns to Drop:
['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
Shape before dropping: (1470, 35)
Shape after dropping: (1470, 31)


### Observation

The dataset was analyzed to identify columns that do not contribute meaningful information for predicting employee attrition. Constant columns and identifier columns were removed because they provide no useful predictive information and may unnecessarily increase model complexity.

In [6]:
# learnt where to use map practically
df["Attrition"] = df["Attrition"].map({
    "Yes": 1,
    "No": 0
})
print(df["Attrition"].head())

0    1
1    0
2    1
3    0
4    0
Name: Attrition, dtype: int64


### Observation

The target variable was converted from categorical values ("Yes" and "No") to numerical values (1 and 0). This transformation is required because machine learning classification algorithms work with numerical target labels.

In [7]:
columns_to_drop=df.select_dtypes(include=["object"]).columns

df=pd.get_dummies(df, columns=columns_to_drop, drop_first=True)
print(df.head(10))

print(df.shape)

   Age  Attrition  DailyRate  DistanceFromHome  Education  \
0   41          1       1102                 1          2   
1   49          0        279                 8          1   
2   37          1       1373                 2          2   
3   33          0       1392                 3          4   
4   27          0        591                 2          1   
5   32          0       1005                 2          2   
6   59          0       1324                 3          3   
7   30          0       1358                24          1   
8   38          0        216                23          3   
9   36          0       1299                27          3   

   EnvironmentSatisfaction  HourlyRate  JobInvolvement  JobLevel  \
0                        2          94               3         2   
1                        3          61               2         2   
2                        4          92               2         1   
3                        4          56               3  

/var/folders/1n/1fwqfjps4v5cmtm5yhyw4kmm0000gn/T/ipykernel_2296/3941537351.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columns_to_drop=df.select_dtypes(include=["object"]).columns


### Observation

All categorical features were converted into numerical features using One-Hot Encoding. This transformation enables machine learning algorithms to process categorical information without introducing artificial ordering between categories. The `drop_first=True` parameter was used to avoid redundant features.

In [8]:
# axis=1 tells that we need to think horizontally and it is a value in a column not the column name itself
df1=df.drop("Attrition",axis=1) 
df2=df["Attrition"]

numeric_columns=df1.select_dtypes(include=["int64", "float64"]).columns

df1[numeric_columns]=StandardScaler().fit_transform(df1[numeric_columns])

print(df1.head(10))



        Age  DailyRate  DistanceFromHome  Education  EnvironmentSatisfaction  \
0  0.446350   0.742527         -1.010909  -0.891688                -0.660531   
1  1.322365  -1.297775         -0.147150  -1.868426                 0.254625   
2  0.008343   1.414363         -0.887515  -0.891688                 1.169781   
3 -0.429664   1.461466         -0.764121   1.061787                 1.169781   
4 -1.086676  -0.524295         -0.887515  -1.868426                -1.575686   
5 -0.539166   0.502054         -0.887515  -0.891688                 1.169781   
6  2.417384   1.292887         -0.764121   0.085049                 0.254625   
7 -0.758170   1.377177          1.827158  -1.868426                 1.169781   
8  0.117845  -1.453958          1.703764   0.085049                 1.169781   
9 -0.101159   1.230910          2.197341   0.085049                 0.254625   

   HourlyRate  JobInvolvement  JobLevel  JobSatisfaction  MonthlyIncome  ...  \
0    1.383138        0.379672 -0.057788

### Observation

The continuous numerical features were standardized using StandardScaler. Standardization ensures that features with larger numerical ranges do not dominate the learning process of machine learning algorithms. The target variable was kept separate and was not scaled.

# Task 3 - Exploratory Data Analysis (EDA)

In [10]:
# checking the columns as we did some preprocessing so original columns got encode
print(df.columns)

Index(['Age', 'Attrition', 'DailyRate', 'DistanceFromHome', 'Education',
       'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel',
       'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
       'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole',
       'YearsSinceLastPromotion', 'YearsWithCurrManager',
       'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely',
       'Department_Research & Development', 'Department_Sales',
       'EducationField_Life Sciences', 'EducationField_Marketing',
       'EducationField_Medical', 'EducationField_Other',
       'EducationField_Technical Degree', 'Gender_Male',
       'JobRole_Human Resources', 'JobRole_Laboratory Technician',
       'JobRole_Manager', 'JobRole_Manufacturing Director',
       'JobRole_Research Director', 'JobRole_Res

In [21]:
# Create a fresh copy of the original dataset for EDA
df_eda = pd.read_csv("../WA_Fn-UseC_-HR-Employee-Attrition.csv")

# just checking the shape of the dataset for EDA
df_eda.shape

(1470, 35)

In [ ]:
# DROPPING THE ALREADY FOUND IRRELEVANT COLUMNS FROM THE DF_EDA DATASET 
df_eda = df_eda.drop(
    columns=[
        "EmployeeCount",
        "EmployeeNumber",
        "Over18",
        "StandardHours"
    ]
)
# VERIFYING 
df_eda.shape


(1470, 31)

In [ ]:
# Point1 - Attrition rate by Department — which department loses the most employees
department_attrition = (
    df_eda.groupby("Department")["Attrition"]
          .apply(lambda x: (x == "Yes").mean() * 100) # remember that to use lamda x like this 
          .sort_values(ascending=False)
)
highest_attrition_department = (df_eda.groupby("Department")["Attrition"]
                                .apply(lambda x: (x == "Yes").mean() * 100) 
                                .sort_values(ascending=False)[:1]
)

print(f"Departments according to attrition rate:\n{department_attrition}\n")
print(f"Column with the highest attrition rate: {highest_attrition_department}")

Departments according to attrition rate:
Department
Sales                     20.627803
Human Resources           19.047619
Research & Development    13.839750
Name: Attrition, dtype: float64

Column with the highest attrition rate: Department
Sales    20.627803
Name: Attrition, dtype: float64


In [ ]:
# Point2 - Attrition rate by Job Role — which roles have the highest exit rate

job_role_attrition = (
    df_eda.groupby("JobRole")["Attrition"]
          .apply(lambda x: (x == "Yes").mean() * 100) # remember that to use lamda x like this 
          .sort_values(ascending=False)
)
highest_attrition_job_role = (df_eda.groupby("JobRole")["Attrition"]
                              .apply(lambda x: (x == "Yes").mean() * 100)
                              .sort_values(ascending=False)[:1]
)
print(f"Job roles according to attrition rate:\n{job_role_attrition}\n")
print(f"Column with the highest attrition rate: {highest_attrition_job_role}")

Job roles according to attrition rate:
JobRole
Sales Representative         39.759036
Laboratory Technician        23.938224
Human Resources              23.076923
Sales Executive              17.484663
Research Scientist           16.095890
Manufacturing Director        6.896552
Healthcare Representative     6.870229
Manager                       4.901961
Research Director             2.500000
Name: Attrition, dtype: float64

Column with the highest attrition rate: JobRole
Sales Representative    39.759036
Name: Attrition, dtype: float64


In [ ]:
# Point3 - Attrition vs Monthly Income — do lower paid employees leave more

# print(df_eda.columns)
'''
MonthlyIncome_attrition = (
    df_eda.groupby("MonthlyIncome")["Attrition"]
          .apply(lambda x: (x == "Yes").mean() * 100) # remember that to use lamda x like this 
          .sort_values(ascending=False)
)

highest_attrition_monthly_income = (df_eda.groupby("MonthlyIncome")["Attrition"]
                                    .apply(lambda x : (x=="Yes).mean() *100)"))
                                    .sort_values(ascending=False)[:1])
print(f"Attrition rate by Monthly Income:\n{MonthlyIncome_attrition}\n")
print(f"Highest attrition rate by Monthly Income: {highest_attrition_monthly_income}")
''' # wrong way as there can be unique salries 

# correct way 
salary_bins = [0, 5000, 10000, 15000, 20000]

salary_labels = [
    "0-5000",
    "5001-10000",
    "10001-15000",
    "15001-20000"
]

# binning the MonthlyIncome column into salary ranges
df_eda["SalaryRange"] = pd.cut(
    df_eda["MonthlyIncome"],
    bins=salary_bins,
    labels=salary_labels
)

MonthlyIncome_attrition = (
    df_eda.groupby("SalaryRange")["Attrition"]
          .apply(lambda x: (x == "Yes").mean() * 100) # remember that to use lamda x like this 
          .sort_values(ascending=False)
)

highest_attrition_monthly_income = (df_eda.groupby("SalaryRange")["Attrition"]
                                    .apply(lambda x : (x=="Yes").mean() *100)
                                    .sort_values(ascending=False)[:1] # or MonthlyIncome_attrition.head(1)
)
print(f"Attrition rate by Monthly Income:\n{MonthlyIncome_attrition}\n")
print(f"Highest attrition rate by Monthly Income: {highest_attrition_monthly_income}")




Attrition rate by Monthly Income:
SalaryRange
0-5000         21.762350
10001-15000    13.513514
5001-10000     11.136364
15001-20000     3.759398
Name: Attrition, dtype: float64

Highest attrition rate by Monthly Income: SalaryRange
0-5000    21.76235
Name: Attrition, dtype: float64


### Observation

- Employees earning between **₹0–5000** have the highest attrition rate (**21.76%**).
- Employees earning between **₹15001–20000** have the lowest attrition rate (**3.76%**).
- Overall, employees in lower salary ranges tend to exhibit higher attrition than those in higher salary ranges.
- The relationship is not perfectly linear, suggesting that salary is an important factor but not the only factor influencing employee attrition.

In [ ]:
# Point4 - Attrition vs Work-Life Balance rating — is there a visible pattern
# print(df_eda.columns)
# print(df_eda["WorkLifeBalance"].value_counts().sort_index())
# print(df_eda["WorkLifeBalance"].nunique())



WorkLifeBalance_attrition = (
    df_eda.groupby("WorkLifeBalance")["Attrition"]
          .apply(lambda x: (x == "Yes").mean() * 100)
          .sort_values(ascending=False)
)

highest_attrition_worklife_balance = WorkLifeBalance_attrition.head(1) # or WorkLifeBalance_attrition.head(1)

print(f"Attrition rate by Work-Life Balance rating:\n{WorkLifeBalance_attrition}\n")
print(f"Highest attrition rate by Work-Life Balance rating: {highest_attrition_worklife_balance}")

4
Attrition rate by Work-Life Balance rating:
WorkLifeBalance
1    31.250000
4    17.647059
2    16.860465
3    14.221725
Name: Attrition, dtype: float64

Highest attrition rate by Work-Life Balance rating: WorkLifeBalance
1    31.25
Name: Attrition, dtype: float64


### Observation

- Employees with a **Work-Life Balance rating of 1** have the highest attrition rate (**31.25%**).
- Employees with higher Work-Life Balance ratings generally exhibit lower attrition rates.
- This suggests that poor work-life balance may be associated with increased employee turnover.
- However, work-life balance should be considered alongside other factors such as salary, job satisfaction, and overtime before drawing conclusions.

In [49]:
# Point5 - Attrition vs Years at Company — at what point in tenure do employees leave most?

# print(df_eda.columns)
# print(df_eda["YearsAtCompany"].max())

years_ranges = [0, 5, 10, 15, 20, 25, 30, 35, 40]
years_labels = ["0-5", "5-10", "10-15", "15-20", "20-25", "25-30", "30-35", "35-40"]

df_eda["yearsranges"]=pd.cut(df_eda["YearsAtCompany"], bins=years_ranges, labels=years_labels)
years_attrition = (
    df_eda.groupby("yearsranges")["Attrition"]
          .apply(lambda x: (x == "Yes").mean() * 100)
          .sort_values(ascending=False)
)

highest_attrition_years = years_attrition.head(1)

print(f"Attrition rate by Years at Company:\n{years_attrition}\n")
print(f"Year range with highest attrition rate: {highest_attrition_years.index[0]}")

Attrition rate by Years at Company:
yearsranges
30-35    25.000000
35-40    25.000000
0-5      19.945355
5-10     12.276786
20-25     9.756098
15-20     6.944444
10-15     6.481481
25-30     0.000000
Name: Attrition, dtype: float64

Year range with highest attrition rate: 30-35


### Observation

- Employees in the **30–35** and **35–40** year experience ranges show the highest attrition rates, but these groups contain relatively few employees and should be interpreted with caution.
- Among the larger employee groups, those with **0–5 years** at the company exhibit a comparatively higher attrition rate.
- This suggests that employee turnover is more common during the early years of employment.

### important

- always conclude after finding the number of employees in that particular range before concluding randomly just by looking at numbers 

## Business Insights

### 1. Department-wise Attrition

The **Sales** department recorded the highest employee attrition rate among all departments. This suggests that employees in Sales are leaving the company more frequently than those in other departments, indicating a potential need for department-specific retention strategies.

### 2. Job Role Analysis

Certain job roles experienced significantly higher attrition than others. These roles should be prioritized for further investigation to identify possible causes such as workload, career growth opportunities, or job satisfaction.

### 3. Monthly Income and Attrition

Employees earning **₹0–₹5,000** per month exhibited the highest attrition rate (**21.76%**), while employees earning **₹15,001–₹20,000** had the lowest attrition rate (**3.76%**). This indicates that lower salary ranges are associated with higher employee turnover.

### 4. Work-Life Balance

Employees with a **Work-Life Balance rating of 1** showed the highest attrition rate (**31.25%**). This suggests that poor work-life balance may be an important factor contributing to employees leaving the organization.

### 5. Years at Company

Employees with **0–5 years** at the company showed comparatively higher attrition among the larger employee groups, indicating that employee turnover is more common during the early stages of employment. Although the **30–35** and **35–40** year groups had the highest percentages, these groups contain relatively few employees and should be interpreted with caution.
